# UBS Kurzzeit-Stresstest für Kapitel 4
**Datenquelle:** UBS Group AG, 2023, S. 89–90  

**Einheit:** USD Mio.

**Methodik:** Lineares Punktschätzungsmodell (Point Estimate)

In [1]:
import pandas as pd
import math

pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', 120)

## Datenbasis

In [2]:
# HQLA (zähler der LCR, nach haircut)
# Quelle: UBS Group AG, 2023, S. 89
HQLA_L1    = 302_112   # level 1 aktiva (ohne haircut)
HQLA_L2    =  29_456   # level 2 aktiva (mit haircut; level 2a = 15 %, level 2b = 50 %)
HQLA_TOTAL = HQLA_L1 + HQLA_L2   # = 331 568

# gewichtete abflüsse & zuflüsse
# Quelle: UBS Group AG, 2023, S. 90
OUTFLOW_GROSS_30D = 390_134   # brutto-abflüsse
INFLOW_30D        = 208_441   # gesamte zuflüsse

# 75 %-cap auf zuflüsse (Basel III, 2019, Art. LCR40): zuflüsse dürfen max. 75 % der abflüsse decken
INFLOW_EFF        = min(INFLOW_30D, 0.75 * OUTFLOW_GROSS_30D)   # cap greift nicht --> 208 441

NET_OUTFLOW_30D   = OUTFLOW_GROSS_30D - INFLOW_EFF   # = 181'693
LCR_RATIO         = HQLA_TOTAL / NET_OUTFLOW_30D     # ≈ 182.5 % (jahresendwert)

print(f'HQLA Level 1:            {HQLA_L1:>10,.0f} Mio. USD')
print(f'HQLA Level 2:            {HQLA_L2:>10,.0f} Mio. USD')
print(f'HQLA Total:              {HQLA_TOTAL:>10,.0f} Mio. USD')
print(f'Brutto-Abflüsse (30d):   {OUTFLOW_GROSS_30D:>10,.0f} Mio. USD')
print(f'Zuflüsse effektiv (30d): {INFLOW_EFF:>10,.0f} Mio. USD')
print(f'Netto-Abflüsse (30d):    {NET_OUTFLOW_30D:>10,.0f} Mio. USD')
print(f'LCR (Jahresendwert):     {LCR_RATIO:>10.1%}')

HQLA Level 1:               302,112 Mio. USD
HQLA Level 2:                29,456 Mio. USD
HQLA Total:                 331,568 Mio. USD
Brutto-Abflüsse (30d):      390,134 Mio. USD
Zuflüsse effektiv (30d):    208,441 Mio. USD
Netto-Abflüsse (30d):       181,693 Mio. USD
LCR (Jahresendwert):         182.5%


## Szenario-Parameter

Jedes Szenario ist durch zwei Parameter definiert:
- **k**: Multiplikator auf die tägliche Brutto-Abflussrate (`Brutto_tägl = OUTFLOW_GROSS_30D / 30 × k`)
- **inflow_f**: Anteil der Zuflüsse, der im Stress noch eintrifft

**Herleitung k:**
- Szenario A (k = 1.0): Standard-LCR; Abflüsse gemäss Basel-III-Vorgabe über 30 Tage
- Szenario B (k = 2.24): CS-Krise März 2023; empirische Tagesrate 7.45 % vs. LCR-Tagesrate 3.3 % → k = 7.45/3.3 ≈ 2.24
- Szenario C (k = e^1.54 ≈ 4.66): Digitaler Flash-Run; Regressions­koeffizient für Log-Zahlungswert von Run-Banken an Stress­tagen

**Herleitung inflow_f:**
- Szenario B (72 %): Run-Banken erhielten im März 2023 ca. 20–28 % weniger Zahlungen von Gegenparteien; 72 % ist leicht konservativ
- Szenario C (0 %): Im akuten Panikumfeld nehme ich an, dass Zuflüsse vollständig gestoppt werden
**HQLA-Basis:**
- Szenario A & B: HQLA_TOTAL (Level-1 + Level-2); Level-2-Monetarisierung realistisch über 10+ Tage
- Szenario C: nur HQLA_L1, da Level-2-Aktiva in Panikphasen nicht innert Stunden monetarisierbar sind

In [3]:
# k-Faktor, Zufluss-Faktor, HQLA-Basis
SCENARIOS = {
    'A Basel III Standard'           : (1.00,           1.00, HQLA_TOTAL),
    'B CS-Benchmark  (k = 2.24)'     : (2.24,           0.72, HQLA_TOTAL),
    'C Digital Flash Run (k = e^1.54)': (math.exp(1.54), 0.00, HQLA_L1),
}

## Modell und Ergebnistabelle

In [4]:
def daily_net_drain(k, inflow_f):
    """konstanter täglicher netto-abfluss: Brutto(skaliert mit k) minus Zuflüsse(reduziert mit f)."""
    return OUTFLOW_GROSS_30D / 30 * k  -  INFLOW_EFF / 30 * inflow_f

def survival_days(hqla_b, drain):
    """tage bis HQLA = 0"""
    d = hqla_b / drain
    return round(d, 1)

rows = []
for name, (k, inf, hqla_b) in SCENARIOS.items():
    drain = daily_net_drain(k, inf)
    surv  = survival_days(hqla_b, drain)
    rows.append({
        'Szenario'                         : name,
        'k'                                : round(k, 2),
        'Zufluss-Faktor'                   : f'{inf:.0%}',
        'HQLA-Basis (Mio. USD)'            : f'{hqla_b:,.0f}',
        'Tägl. Brutto-Abfluss (Mio. USD)'  : f'{OUTFLOW_GROSS_30D/30*k:,.0f}',
        'Tägl. Netto-Abfluss (Mio. USD)'   : f'{drain:,.0f}',
        'Überlebensdauer (Tage)'           : str(surv),
    })

df_main = pd.DataFrame(rows)
print('Tabelle 2: Szenario-Ergebnisse')
print(df_main.to_string(index=False))

Tabelle 2: Szenario-Ergebnisse
                        Szenario    k Zufluss-Faktor HQLA-Basis (Mio. USD) Tägl. Brutto-Abfluss (Mio. USD) Tägl. Netto-Abfluss (Mio. USD) Überlebensdauer (Tage)
            A Basel III Standard 1.00           100%               331,568                          13,004                          6,056                   54.7
      B CS-Benchmark  (k = 2.24) 2.24            72%               331,568                          29,130                         24,127                   13.7
C Digital Flash Run (k = e^1.54) 4.66             0%               302,112                          60,661                         60,661                    5.0


## Sensitivitätsanalyse

In [5]:
# variation: k von 1.0 bis 5.0 * zufluss-szenario (100 %, 72 %, 0 %)
# HQLA-Basis: HQLA_TOTAL solange k < e^{1.54}, sonst nur HQLA_L1

k_vals    = [1.0, 1.1, 1.2, 1.3, 1.4, 1.5, 2.0, 2.24, 2.5, 3.0, 3.5, 4.0, round(math.exp(1.54), 2), 5.0]
inf_cases = {'Zuflüsse 100 %': 1.0, 'Zuflüsse 72 %': 0.72, 'Zuflüsse 0 %': 0.0}

sens_rows = []
for k in k_vals:
    hqla_b = HQLA_L1 if k >= round(math.exp(1.54), 2) else HQLA_TOTAL
    row = {'k': round(k, 2), 'HQLA-Basis': 'L1+L2' if hqla_b == HQLA_TOTAL else 'L1 only'}
    for label, inf in inf_cases.items():
        drain = daily_net_drain(k, inf)
        row[label] = str(survival_days(hqla_b, drain))
    sens_rows.append(row)

df_sens = pd.DataFrame(sens_rows)
print('Tabelle 3: Sensitivitätsanalyse - Überlebensdauer in Tagen')
print(df_sens.to_string(index=False))

Tabelle 3: Sensitivitätsanalyse - Überlebensdauer in Tagen
   k HQLA-Basis Zuflüsse 100 % Zuflüsse 72 % Zuflüsse 0 %
1.00      L1+L2           54.7          41.4         25.5
1.10      L1+L2           45.1          35.6         23.2
1.20      L1+L2           38.3          31.3         21.2
1.30      L1+L2           33.3          27.9         19.6
1.40      L1+L2           29.5          25.1         18.2
1.50      L1+L2           26.4          22.9         17.0
2.00      L1+L2           17.4          15.8         12.7
2.24      L1+L2           14.9          13.7         11.4
2.50      L1+L2           13.0          12.1         10.2
3.00      L1+L2           10.3           9.7          8.5
3.50      L1+L2            8.6           8.2          7.3
4.00      L1+L2            7.4           7.1          6.4
4.66    L1 only            5.6           5.4          5.0
5.00    L1 only            5.2           5.0          4.6


## Sensitivitätsanalyse 2: HQLA-Basis als Variable

frage: wie viel mehr HQLA müsste die UBS halten, um 30 tage zu überstehen?

In [6]:
# benötigte HQLA für 30-Tage-überleben
print('benötigte HQLA für 30-tage-überleben:')
for name, k, inf in [('A', 1.0, 1.0),( 'B', 2.24, 0.72),('C', math.exp(1.54), 0.0)]:
    d = daily_net_drain(k, inf)
    needed = d * 30
    mult = needed / HQLA_TOTAL
    print(f'  szenario {name}: {needed:>12,.0f} mio. USD  ({mult:.1f}× aktueller bestand)')

print()
mult_vals = [1.0, 1.05, 1.1, 1.2, 1.5, 2.0, 2.5, 3.0, 5.0, 6.0]
SCEN2 = [('A – standard', 1.00, 1.0, 'total'),('B – CS-benchmark', 2.24, 0.72, 'total'),('C – flash run', math.exp(1.54), 0.0, 'l1')]
rows_hqla = []
for m in mult_vals:
    row = {'multiplikator': f'{m:.2f}x', 'HQLA absolut': f'{HQLA_TOTAL*m:,.0f}'}
    for name, k, inf, base in SCEN2:
        hb = (HQLA_L1 if base=='l1' else HQLA_TOTAL) * m
        d = daily_net_drain(k, inf)
        row[name] = str(survival_days(hb, d))
    rows_hqla.append(row)

print(pd.DataFrame(rows_hqla).to_string(index=False))
print('\nfazit: szenario B erst ab 3×, szenario C erst ab ca. 6× aktueller HQLA überlebensfähig.')

benötigte HQLA für 30-tage-überleben:
  szenario A:      181,693 mio. USD  (0.5× aktueller bestand)
  szenario B:      723,823 mio. USD  (2.2× aktueller bestand)
  szenario C:    1,819,815 mio. USD  (5.5× aktueller bestand)

multiplikator HQLA absolut A – standard B – CS-benchmark C – flash run
        1.00x      331,568         54.7             13.7           5.0
        1.05x      348,146         57.5             14.4           5.2
        1.10x      364,725         60.2             15.1           5.5
        1.20x      397,882         65.7             16.5           6.0
        1.50x      497,352         82.1             20.6           7.5
        2.00x      663,136        109.5             27.5          10.0
        2.50x      828,920        136.9             34.4          12.5
        3.00x      994,704        164.2             41.2          14.9
        5.00x    1,657,840        273.7             68.7          24.9
        6.00x    1,989,408        328.5             82.5         